In [27]:

from pathlib import Path
import json
import warnings

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display, Markdown

from sklearn.model_selection import train_test_split, KFold, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    mean_absolute_percentage_error
)

try:
    from xgboost import XGBRegressor
except ImportError as e:
    raise ImportError(
        "Chưa cài xgboost. Hãy chạy cell: %pip install xgboost openpyxl joblib -q"
    ) from e

warnings.filterwarnings("ignore")

# Cấu hình hiển thị
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)

RANDOM_STATE = 42
TEST_SIZE = 0.2
CV_SPLITS = 5

# MÔ HÌNH NÂNG CAO: RANDOM FOREST VÀ XGBOOST

## Vì sao chọn hai mô hình này?

### 1. Random Forest Regressor
- Phù hợp với dữ liệu dạng bảng (tabular data)
- Bắt được quan hệ phi tuyến giữa đặc trưng và giá thuê
- Ổn định hơn Decision Tree đơn lẻ
- Ít bị overfitting hơn so với một cây quyết định riêng lẻ

### 2. XGBoost Regressor
- Là một trong những mô hình mạnh nhất cho dữ liệu bảng
- Học tốt các tương tác phức tạp giữa nhiều biến đầu vào
- Thường cho độ chính xác cao trong bài toán hồi quy thực tế

## Liên hệ với dataset PhongTro
Kết quả EDA cho thấy tương quan giữa diện tích và giá thuê là khoảng 0.55,
tức có mối liên hệ dương nhưng không hoàn toàn tuyến tính.
Vì vậy, nhóm lựa chọn Random Forest và XGBoost để so sánh,
nhằm khai thác tốt hơn quan hệ phi tuyến và tương tác giữa các đặc trưng.


In [28]:
# DEFINE PATHS
MANUAL_ANALYSIS_ROOT = r'D:\DOHOANGPHUC\UTH\TriTueNhanTaovaUngDung\Project2\TR-TU-NH-N-T-O-NG-D-NG-\Analysis'

def find_analysis_root(start_path: Path, manual_root=None) -> Path:
    """
    Tìm thư mục gốc 'Analysis' dựa trên vị trí đang chạy notebook.
    Yêu cầu thư mục đó phải chứa:
    - Dataset
    - outputs
    """
    if manual_root is not None:
        root = Path(manual_root)
        if not root.exists():
            raise FileNotFoundError(f"MANUAL_ANALYSIS_ROOT không tồn tại: {root}")
        return root

    start_path = start_path.resolve()
    candidates = [start_path] + list(start_path.parents)

    for p in candidates:
        if (p / "Dataset").exists() and (p / "outputs").exists():
            return p

    raise FileNotFoundError(
        "Không tìm thấy thư mục gốc 'Analysis'. "
        "Hãy gán MANUAL_ANALYSIS_ROOT bằng đường dẫn tuyệt đối đến thư mục Analysis."
    )

ANALYSIS_ROOT = find_analysis_root(Path.cwd(), MANUAL_ANALYSIS_ROOT)

RAW_DATA_PATH = ANALYSIS_ROOT / "Dataset" / "raw" / "Maindataset" / "PhongTro.xlsx"
CLEAN_DATA_PATH = ANALYSIS_ROOT / "outputs" / "data" / "phongtro_cleaned.csv"

OUTPUT_ROOT = ANALYSIS_ROOT / "outputs"
DATA_OUTPUT_DIR = OUTPUT_ROOT / "data"
MODEL_DIR = OUTPUT_ROOT / "models"
FIGURE_DIR = OUTPUT_ROOT / "figures"
REPORT_DIR = OUTPUT_ROOT / "reports"

for folder in [DATA_OUTPUT_DIR, MODEL_DIR, FIGURE_DIR, REPORT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("ANALYSIS_ROOT:", ANALYSIS_ROOT)
print("RAW_DATA_PATH:", RAW_DATA_PATH)
print("CLEAN_DATA_PATH:", CLEAN_DATA_PATH)
print("MODEL_DIR:", MODEL_DIR)
print("FIGURE_DIR:", FIGURE_DIR)
print("REPORT_DIR:", REPORT_DIR)

ANALYSIS_ROOT: D:\DOHOANGPHUC\UTH\TriTueNhanTaovaUngDung\Project2\TR-TU-NH-N-T-O-NG-D-NG-\Analysis
RAW_DATA_PATH: D:\DOHOANGPHUC\UTH\TriTueNhanTaovaUngDung\Project2\TR-TU-NH-N-T-O-NG-D-NG-\Analysis\Dataset\raw\Maindataset\PhongTro.xlsx
CLEAN_DATA_PATH: D:\DOHOANGPHUC\UTH\TriTueNhanTaovaUngDung\Project2\TR-TU-NH-N-T-O-NG-D-NG-\Analysis\outputs\data\phongtro_cleaned.csv
MODEL_DIR: D:\DOHOANGPHUC\UTH\TriTueNhanTaovaUngDung\Project2\TR-TU-NH-N-T-O-NG-D-NG-\Analysis\outputs\models
FIGURE_DIR: D:\DOHOANGPHUC\UTH\TriTueNhanTaovaUngDung\Project2\TR-TU-NH-N-T-O-NG-D-NG-\Analysis\outputs\figures
REPORT_DIR: D:\DOHOANGPHUC\UTH\TriTueNhanTaovaUngDung\Project2\TR-TU-NH-N-T-O-NG-D-NG-\Analysis\outputs\reports


In [29]:
# LOAD DATA

if CLEAN_DATA_PATH.exists():
    df = pd.read_csv(CLEAN_DATA_PATH)
    data_source = CLEAN_DATA_PATH
elif RAW_DATA_PATH.exists():
    df = pd.read_excel(RAW_DATA_PATH, engine="openpyxl")
    data_source = RAW_DATA_PATH
else:
    raise FileNotFoundError(
        "Không tìm thấy cả 'phongtro_cleaned.csv' lẫn 'PhongTro.xlsx'. "
        "Hãy kiểm tra lại cấu trúc thư mục."
    )

df.columns = [str(c).strip() for c in df.columns]

print(f"Đã đọc dữ liệu từ: {data_source}")
print(f"Kích thước dữ liệu: {df.shape}")

display(df.head())
display(df.info())
display(df.describe(include="all").T.head(20))

Đã đọc dữ liệu từ: D:\DOHOANGPHUC\UTH\TriTueNhanTaovaUngDung\Project2\TR-TU-NH-N-T-O-NG-D-NG-\Analysis\outputs\data\phongtro_cleaned.csv
Kích thước dữ liệu: (1368, 20)


,tieude,dientich,giavnd,vitri,phanloai,sophong,area_m2,price_vnd,standardized_location,property_type_clean,price_per_m2,price_group,area_group,has_studio,has_balcony,has_mezzanine,has_furniture,has_new,has_window,has_elevator
0,Studio thang máy Nguyễn Thị Minh Khai gần Thảo...,40 m,8000000,Quận 1,PhongTro,1,40.0,8000000.0,Quận 1,Phòng trọ,200000.000000,Trung bình (4-8.5 triệu),Vừa (25-50m²),True,False,False,False,False,False,True
1,Khai trương toà nhà mới Ngã 6 Phù Đổng,30 m,8500000,Quận 1,PhongTro,1,30.0,8500000.0,Quận 1,Phòng trọ,283333.333333,Trung bình (4-8.5 triệu),Vừa (25-50m²),False,False,False,False,True,False,False
2,Phòng ban công giá rẻ gần Lý Chính Thắng,28 m,5500000,Quận 1,PhongTro,1,28.0,5500000.0,Quận 1,Phòng trọ,196428.571429,Trung bình (4-8.5 triệu),Vừa (25-50m²),False,True,False,False,False,False,False
3,Phòng Studio đẹp khu Hoàng Sa,28 m,8000000,Quận 1,PhongTro,1,28.0,8000000.0,Quận 1,Phòng trọ,285714.285714,Trung bình (4-8.5 triệu),Vừa (25-50m²),True,False,False,False,False,False,False
4,Phòng ban công rẻ bờ kè Hoàng Sa,25 m,5200000,Quận 1,PhongTro,1,25.0,5200000.0,Quận 1,Phòng trọ,208000.000000,Trung bình (4-8.5 triệu),Nhỏ (<25m²),False,True,False,False,False,False,False


<class 'pandas.DataFrame'>
RangeIndex: 1368 entries, 0 to 1367
Data columns (total 20 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   tieude                 1368 non-null   str    
 1   dientich               1368 non-null   str    
 2   giavnd                 1368 non-null   int64  
 3   vitri                  1368 non-null   str    
 4   phanloai               1368 non-null   str    
 5   sophong                1368 non-null   int64  
 6   area_m2                1368 non-null   float64
 7   price_vnd              1368 non-null   float64
 8   standardized_location  1368 non-null   str    
 9   property_type_clean    1368 non-null   str    
 10  price_per_m2           1368 non-null   float64
 11  price_group            1368 non-null   str    
 12  area_group             1368 non-null   str    
 13  has_studio             1368 non-null   bool   
 14  has_balcony            1368 non-null   bool   
 15  has_mezzanine  

None

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
tieude,1368,1329,Chung cư Hoa Sen Lạc Long Quân Quận 11,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN
dientich,1368,121,30 m,214,NaN,NaN,NaN,NaN,NaN,NaN,NaN
giavnd,1368.0,NaN,NaN,NaN,7435455.282895,7672839.242416,500000.0,4000000.0,5700000.0,8500000.0,100000000.0
vitri,1368,43,Quận 1,60,NaN,NaN,NaN,NaN,NaN,NaN,NaN
phanloai,1368,10,PhongTro,465,NaN,NaN,NaN,NaN,NaN,NaN,NaN
sophong,1368.0,NaN,NaN,NaN,1.319444,0.639703,1.0,1.0,1.0,1.0,3.0
area_m2,1368.0,NaN,NaN,NaN,44.176901,34.048375,8.0,25.0,34.0,50.0,300.0
price_vnd,1368.0,NaN,NaN,NaN,7435455.282895,7672839.242416,500000.0,4000000.0,5700000.0,8500000.0,100000000.0
standardized_location,1368,24,Quận 1,80,NaN,NaN,NaN,NaN,NaN,NaN,NaN
property_type_clean,1368,10,Phòng trọ,465,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [30]:
# IDENTIFY TARGET & CLEAN FEATURES

TARGET_CANDIDATES = [
    "GiaVND",
    "gia_vnd",
    "gia_vnd_clean",
    "price_vnd",
    "gia_thue_vnd",
    "target"
]

def find_target_column(columns, candidates):
    lower_map = {str(c).lower(): c for c in columns}

    # Ưu tiên khớp chính xác
    for cand in candidates:
        if cand.lower() in lower_map:
            return lower_map[cand.lower()]

    # Nếu không có thì tìm gần đúng
    for col in columns:
        col_lower = str(col).lower()
        if "giavnd" in col_lower or ("gia" in col_lower and "vnd" in col_lower):
            return col

    raise ValueError(
        "Không tìm thấy cột target. "
        "Hãy kiểm tra lại tên cột giá thuê, ví dụ: GiaVND."
    )

target_col = find_target_column(df.columns, TARGET_CANDIDATES)
print("Target column =", target_col)

# Nếu target chưa phải số, cố gắng convert
if not pd.api.types.is_numeric_dtype(df[target_col]):
    df[target_col] = (
        df[target_col]
        .astype(str)
        .str.replace(r"[^\d.\-]", "", regex=True)
        .replace("", np.nan)
    )
    df[target_col] = pd.to_numeric(df[target_col], errors="coerce")

# Xóa dòng thiếu target
before_rows = len(df)
df = df.dropna(subset=[target_col]).copy()
after_rows = len(df)
print(f"Số dòng bị loại do thiếu target: {before_rows - after_rows}")

# Các cột không nên đưa vào model
# - id/url/link/title/description: thường không giúp dự đoán tốt, lại gây nhiễu
# - các cột chứa 'gia' hoặc 'price' khác target: dễ gây leakage
manual_drop_candidates = [
    "Unnamed: 0",
    "id", "ID",
    "url", "URL",
    "link", "Link",
    "title", "Title", "tieu_de", "TieuDe",
    "description", "Description", "mo_ta", "MoTa",
    "dia_chi_chi_tiet", "address_detail", "address"
]

existing_manual_drop_cols = [c for c in manual_drop_candidates if c in df.columns]

leakage_cols = [
    c for c in df.columns
    if c != target_col and any(k in str(c).lower() for k in ["gia", "price"])
]

drop_cols = sorted(set(existing_manual_drop_cols + leakage_cols))

print("Các cột sẽ loại bỏ khỏi feature set:")
print(drop_cols if drop_cols else "Không có cột nào cần loại bỏ thêm.")

X = df.drop(columns=[target_col] + drop_cols, errors="ignore").copy()
y = df[target_col].copy()

# Xóa các cột hằng số (không có giá trị dự báo)
constant_cols = [c for c in X.columns if X[c].nunique(dropna=False) <= 1]
if constant_cols:
    X = X.drop(columns=constant_cols)
    print("Đã loại các cột hằng số:", constant_cols)

print("Shape X:", X.shape)
print("Shape y:", y.shape)

display(X.head())

Target column = giavnd
Số dòng bị loại do thiếu target: 0
Các cột sẽ loại bỏ khỏi feature set:
['price_group', 'price_per_m2', 'price_vnd']
Shape X: (1368, 16)
Shape y: (1368,)


,tieude,dientich,vitri,phanloai,sophong,area_m2,standardized_location,property_type_clean,area_group,has_studio,has_balcony,has_mezzanine,has_furniture,has_new,has_window,has_elevator
0,Studio thang máy Nguyễn Thị Minh Khai gần Thảo...,40 m,Quận 1,PhongTro,1,40.0,Quận 1,Phòng trọ,Vừa (25-50m²),True,False,False,False,False,False,True
1,Khai trương toà nhà mới Ngã 6 Phù Đổng,30 m,Quận 1,PhongTro,1,30.0,Quận 1,Phòng trọ,Vừa (25-50m²),False,False,False,False,True,False,False
2,Phòng ban công giá rẻ gần Lý Chính Thắng,28 m,Quận 1,PhongTro,1,28.0,Quận 1,Phòng trọ,Vừa (25-50m²),False,True,False,False,False,False,False
3,Phòng Studio đẹp khu Hoàng Sa,28 m,Quận 1,PhongTro,1,28.0,Quận 1,Phòng trọ,Vừa (25-50m²),True,False,False,False,False,False,False
4,Phòng ban công rẻ bờ kè Hoàng Sa,25 m,Quận 1,PhongTro,1,25.0,Quận 1,Phòng trọ,Nhỏ (<25m²),False,True,False,False,False,False,False


In [31]:
# TRAIN / TEST SPLIT

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE
)

print("Train shape:", X_train.shape, y_train.shape)
print("Test shape :", X_test.shape, y_test.shape)

# Tách numeric / categorical columns
numeric_cols = X_train.select_dtypes(include=[np.number, "bool"]).columns.tolist()
categorical_cols = [c for c in X_train.columns if c not in numeric_cols]

print("Số cột numeric     :", len(numeric_cols))
print("Số cột categorical :", len(categorical_cols))

print("\nNumeric columns:")
print(numeric_cols)

print("\nCategorical columns:")
print(categorical_cols)

# Pipeline tiền xử lý
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_cols),
        ("cat", categorical_transformer, categorical_cols)
    ],
    remainder="drop"
)

display(Markdown("""
### Giải thích preprocessing
- **Numeric**: điền khuyết bằng median để giảm ảnh hưởng của ngoại lệ
- **Categorical**: điền khuyết bằng giá trị xuất hiện nhiều nhất, sau đó One-Hot Encoding
- **Không scale dữ liệu** vì Random Forest và XGBoost là mô hình cây
"""))

Train shape: (1094, 16) (1094,)
Test shape : (274, 16) (274,)
Số cột numeric     : 9
Số cột categorical : 7

Numeric columns:
['sophong', 'area_m2', 'has_studio', 'has_balcony', 'has_mezzanine', 'has_furniture', 'has_new', 'has_window', 'has_elevator']

Categorical columns:
['tieude', 'dientich', 'vitri', 'phanloai', 'standardized_location', 'property_type_clean', 'area_group']



### Giải thích preprocessing
- **Numeric**: điền khuyết bằng median để giảm ảnh hưởng của ngoại lệ
- **Categorical**: điền khuyết bằng giá trị xuất hiện nhiều nhất, sau đó One-Hot Encoding
- **Không scale dữ liệu** vì Random Forest và XGBoost là mô hình cây


In [32]:
# 6. DEFINE MODELS & PARAM GRIDS

rf_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(
        random_state=RANDOM_STATE,
        n_jobs=-1
    ))
])

xgb_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", XGBRegressor(
        objective="reg:squarederror",
        random_state=RANDOM_STATE,
        n_jobs=-1,
        tree_method="hist",
        eval_metric="rmse"
    ))
])

# Grid vừa phải để tối ưu thời gian chạy nhưng vẫn có ý nghĩa so sánh
rf_param_grid = {
    "model__n_estimators": [200, 400],
    "model__max_depth": [None, 10, 20],
    "model__min_samples_split": [2, 5],
    "model__min_samples_leaf": [1, 2],
    "model__max_features": ["sqrt", 0.8]
}

xgb_param_grid = {
    "model__n_estimators": [300, 500],
    "model__max_depth": [4, 6],
    "model__learning_rate": [0.05, 0.1],
    "model__subsample": [0.8, 1.0],
    "model__colsample_bytree": [0.8, 1.0]
}

model_configs = {
    "RandomForest": {
        "pipeline": rf_pipeline,
        "param_grid": rf_param_grid
    },
    "XGBoost": {
        "pipeline": xgb_pipeline,
        "param_grid": xgb_param_grid
    }
}

display(Markdown("""
### Giải thích tuning
- **Random Forest**:
  - `n_estimators`: số lượng cây
  - `max_depth`: độ sâu tối đa
  - `min_samples_split`, `min_samples_leaf`: kiểm soát overfitting
  - `max_features`: số đặc trưng được xét tại mỗi node

- **XGBoost**:
  - `n_estimators`: số vòng boosting
  - `max_depth`: độ sâu cây
  - `learning_rate`: tốc độ học
  - `subsample`, `colsample_bytree`: giảm overfitting
"""))


### Giải thích tuning
- **Random Forest**:
  - `n_estimators`: số lượng cây
  - `max_depth`: độ sâu tối đa
  - `min_samples_split`, `min_samples_leaf`: kiểm soát overfitting
  - `max_features`: số đặc trưng được xét tại mỗi node

- **XGBoost**:
  - `n_estimators`: số vòng boosting
  - `max_depth`: độ sâu cây
  - `learning_rate`: tốc độ học
  - `subsample`, `colsample_bytree`: giảm overfitting


In [33]:
# EVALUATION FUNCTION

def evaluate_regression(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    mape = mean_absolute_percentage_error(y_true, y_pred) * 100

    return {
        "mae": mae,
        "rmse": rmse,
        "r2": r2,
        "mape_pct": mape
    }

def format_metrics(metrics_dict):
    return {k: round(v, 4) for k, v in metrics_dict.items()}

In [ ]:
# TRAINING & TUNING

cv = KFold(
    n_splits=CV_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE
)

tuning_results = []
tuned_models = {}
prediction_store = {}

for model_name, config in model_configs.items():
    display(Markdown(f"## Đang huấn luyện: {model_name}"))

    grid = GridSearchCV(
        estimator=config["pipeline"],
        param_grid=config["param_grid"],
        scoring="neg_root_mean_squared_error",
        cv=cv,
        n_jobs=-1,
        verbose=1,
        return_train_score=True
    )

    grid.fit(X_train, y_train)

    best_estimator = grid.best_estimator_
    best_params = grid.best_params_
    best_cv_rmse = -grid.best_score_

    y_pred = best_estimator.predict(X_test)
    metrics = evaluate_regression(y_test, y_pred)

    tuning_results.append({
        "model": model_name,
        "best_cv_rmse": best_cv_rmse,
        "test_mae": metrics["mae"],
        "test_rmse": metrics["rmse"],
        "test_r2": metrics["r2"],
        "test_mape_pct": metrics["mape_pct"]
    })

    tuned_models[model_name] = {
        "best_estimator": best_estimator,
        "best_params": best_params,
        "best_cv_rmse": best_cv_rmse
    }

    pred_df = pd.DataFrame({
        "Actual": y_test.values,
        "Predicted": y_pred
    })
    prediction_store[model_name] = pred_df

    safe_name = model_name.lower().replace(" ", "_")

    # Lưu kết quả CV chi tiết
    cv_results_df = pd.DataFrame(grid.cv_results_).sort_values("rank_test_score")
    cv_results_path = REPORT_DIR / f"{safe_name}_gridsearch_results.csv"
    cv_results_df.to_csv(cv_results_path, index=False)

    # Lưu prediction
    pred_path = DATA_OUTPUT_DIR / f"{safe_name}_test_predictions.csv"
    pred_df.to_csv(pred_path, index=False)

    # Lưu model
    model_path = MODEL_DIR / f"{safe_name}_best_model.joblib"
    joblib.dump(best_estimator, model_path)

    # Lưu params
    params_path = REPORT_DIR / f"{safe_name}_best_params.json"
    with open(params_path, "w", encoding="utf-8") as f:
        json.dump(best_params, f, ensure_ascii=False, indent=4, default=str)

    print(f"Best params for {model_name}:")
    print(best_params)

    print(f"Best CV RMSE ({model_name}): {best_cv_rmse:.4f}")
    print(f"Test metrics ({model_name}): {format_metrics(metrics)}")
    print("-" * 80)

## Đang huấn luyện: RandomForest

Fitting 5 folds for each of 48 candidates, totalling 240 fits


In [ ]:
# COMPARE RESULTS

tuning_results_df = pd.DataFrame(tuning_results).sort_values(
    by="test_rmse",
    ascending=True
).reset_index(drop=True)

display(Markdown("# BẢNG SO SÁNH KẾT QUẢ"))
display(tuning_results_df)

comparison_path = REPORT_DIR / "advanced_model_comparison.csv"
tuning_results_df.to_csv(comparison_path, index=False)

best_model_name = tuning_results_df.loc[0, "model"]
best_model = tuned_models[best_model_name]["best_estimator"]

print("Mô hình tốt nhất trên tập test:", best_model_name)
print("Đã lưu bảng so sánh tại:", comparison_path)

In [ ]:
# ACTUAL VS PREDICTED PLOT

best_pred_df = prediction_store[best_model_name].copy()

plt.figure(figsize=(7, 6))
plt.scatter(best_pred_df["Actual"], best_pred_df["Predicted"], alpha=0.6)

min_val = min(best_pred_df["Actual"].min(), best_pred_df["Predicted"].min())
max_val = max(best_pred_df["Actual"].max(), best_pred_df["Predicted"].max())

plt.plot([min_val, max_val], [min_val, max_val], linestyle="--")
plt.xlabel("Actual Price (VND)")
plt.ylabel("Predicted Price (VND)")
plt.title(f"Actual vs Predicted - {best_model_name}")
plt.tight_layout()

scatter_path = FIGURE_DIR / f"{best_model_name.lower()}_actual_vs_predicted.png"
plt.savefig(scatter_path, dpi=300, bbox_inches="tight")
plt.show()

print("Đã lưu biểu đồ tại:", scatter_path)

In [ ]:
# FEATURE IMPORTANCE
def get_feature_names_from_preprocessor(fitted_preprocessor):
    """
    Lấy tên đặc trưng sau khi đã qua ColumnTransformer + OneHotEncoder
    """
    try:
        return fitted_preprocessor.get_feature_names_out()
    except Exception:
        feature_names = []

        for name, transformer, cols in fitted_preprocessor.transformers_:
            if name == "remainder":
                continue

            if isinstance(transformer, Pipeline):
                last_step = transformer.steps[-1][1]

                if hasattr(last_step, "get_feature_names_out"):
                    try:
                        names = last_step.get_feature_names_out(cols)
                    except Exception:
                        names = last_step.get_feature_names_out()
                    feature_names.extend(names)
                else:
                    feature_names.extend(cols)
            else:
                feature_names.extend(cols)

        return np.array(feature_names)

fitted_preprocessor = best_model.named_steps["preprocessor"]
feature_names = get_feature_names_from_preprocessor(fitted_preprocessor)
importances = best_model.named_steps["model"].feature_importances_

feature_importance_df = pd.DataFrame({
    "feature": feature_names,
    "importance": importances
}).sort_values("importance", ascending=False)

display(Markdown(f"# TOP 20 FEATURE IMPORTANCE - {best_model_name}"))
display(feature_importance_df.head(20))

fi_csv_path = REPORT_DIR / f"{best_model_name.lower()}_feature_importance.csv"
feature_importance_df.to_csv(fi_csv_path, index=False)

top20 = feature_importance_df.head(20).sort_values("importance", ascending=True)

plt.figure(figsize=(10, 8))
plt.barh(top20["feature"], top20["importance"])
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.title(f"Top 20 Feature Importance - {best_model_name}")
plt.tight_layout()

fi_fig_path = FIGURE_DIR / f"{best_model_name.lower()}_feature_importance.png"
plt.savefig(fi_fig_path, dpi=300, bbox_inches="tight")
plt.show()

print("Đã lưu file importance CSV tại:", fi_csv_path)
print("Đã lưu biểu đồ importance tại:", fi_fig_path)